<a href="https://colab.research.google.com/github/Savidilsh/vggt/blob/chamudi_R/VGGT_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clone the repo...

In [ ]:
!git clone https://github.com/facebookresearch/vggt.git

Cloning into 'vggt'...
remote: Enumerating objects: 1265, done.
remote: Total 1265 (delta 0), reused 0 (delta 0), pack-reused 1265 (from 1)
Receiving objects: 100% (1265/1265), 64.94 MiB | 42.12 MiB/s, done.
Resolving deltas: 100% (579/579), done.


Go to repo...

In [ ]:
%cd vggt

/content/vggt


Install dependancies...

In [1]:

!pip install -U pip
!pip install open3d -q
!pip install pycolmap
!pip install -q git+https://github.com/cvg/LightGlue.git
!pip install -U \
  hydra-core \
  omegaconf \
  einops \
  yacs \
  scipy \
  opencv-python-headless \
  tqdm \
  matplotlib
!pip install -U "numpy>=2.0.0" "torch>=2.5.0" "torchvision>=0.20.0" "huggingface_hub" "pillow"

import torch, numpy as np
print(f"✓ Torch: {torch.__version__}")
print(f"✓ NumPy: {np.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 64.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 119.0 MB/s  0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [hydra-core]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 162.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 16.1 MB/s  0:00:22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 36.2 MB/s  0:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

✓ Torch: 2.9.0+cu128
✓ NumPy: 2.0.2
✓ CUDA available: True
✓ GPU: Tesla T4
✓ VRAM: 15.8 GB


Direct to VGGT if restarted. if not don't run...

In [ ]:
#if restarted
%cd vggt


/content/vggt


Install vggt without dependencies...

In [4]:
!pip install --no-deps git+https://github.com/facebookresearch/vggt.git

  Cloning https://github.com/facebookresearch/vggt.git to /tmp/pip-req-build-u5e34jzx
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/vggt.git /tmp/pip-req-build-u5e34jzx
  Resolved https://github.com/facebookresearch/vggt.git to commit 44b3afbd1869d8bde4894dd8ea1e293112dd5eba
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for vggt: filename=vggt-0.0.1-py3-none-any.whl size=94100 sha256=7f529c013c8634e84befe83ce2726ebbffb4c490772e853c70274759bec3fcb8
  Stored in directory: /tmp/pip-ephem-wheel-cache-8fsfu3qm/wheels/d5/54/2d/cea5a623f69428042fe8a81d6815b4596c266cc4648a7e74dd
Successfully built vggt


Upload Images...

In [12]:

from google.colab import files
import os

os.makedirs("images", exist_ok=True)

print("Please select your images (.png or .jpg)")
uploaded = files.upload()

for fname in uploaded.keys():
    os.rename(fname, f"images/{fname}")

print(f"\n✓ Uploaded {len(uploaded)} images:")
!ls -1 images | head -10


Please select your images (.png or .jpg)


Saving 001.png to 001.png
Saving 002.png to 002.png
Saving 003.png to 003.png
Saving 004.png to 004.png
Saving 005.png to 005.png

✓ Uploaded 5 images:
001.png
002.png
003.png
004.png
005.png


Run to unload GPU...

In [2]:

import torch, gc
gc.collect()
torch.cuda.empty_cache()


Run to see if there any ongoing tasks on GPU...

In [ ]:
!nvidia-smi


Sun Oct 19 18:03:32 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             28W /   70W |    9248MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

To verify images are in relevent folder...

In [ ]:
%cd images
%ls
%cd ..



/content/images
templeSR0001.png  templeSR0005.png  templeSR0009.png  templeSR0013.png
templeSR0002.png  templeSR0006.png  templeSR0010.png  templeSR0014.png
templeSR0003.png  templeSR0007.png  templeSR0011.png  templeSR0015.png
templeSR0004.png  templeSR0008.png  templeSR0012.png  templeSR0016.png
/content


VGGT reconstructing...

In [9]:

import os, time, glob
import numpy as np
import torch
import open3d as o3d

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

# Configuration
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Using {device.upper()}")


IMAGE_DIR        = "images"
OUT_PLY          = "vggt_reconstruction_global_aligned.ply"
SAVE_POSES_NPY   = True


CONF_THRES       = 7.0
WINDOW_SIZE      = 5
WINDOW_OVERLAP   = 5
VOXEL_SIZE       = 0.0015
MAX_POINTS_FRAME = 250_000
RM_OUTLIER_K     = 40
RM_OUTLIER_STD   = 1.0

USE_FP16_DEPTH   = True

def to_numpy_squeezed(t: torch.Tensor) -> np.ndarray:
    return np.squeeze(t.detach().cpu().numpy())

def ensure_4x4_extrinsics(extrinsic):
    """Convert [S,3,4] or [3,4] or [4,4] → [S,4,4]."""
    extrinsic = np.array(extrinsic)
    if extrinsic.ndim == 2 and extrinsic.shape == (3,4):
        extrinsic = np.vstack([extrinsic, [0,0,0,1]])[None, ...]
    elif extrinsic.ndim == 3 and extrinsic.shape[-2:] == (3,4):
        bottom = np.tile([[0,0,0,1]], (extrinsic.shape[0],1,1))
        extrinsic = np.concatenate([extrinsic, bottom], axis=1)
    elif extrinsic.ndim == 2 and extrinsic.shape == (4,4):
        extrinsic = extrinsic[None, ...]
    assert extrinsic.ndim == 3 and extrinsic.shape[-2:] == (4,4), f"Bad extrinsic shape {extrinsic.shape}"
    return extrinsic

def ensure_3x3_intrinsics(intrinsic):
    intrinsic = np.array(intrinsic)
    if intrinsic.ndim == 2:
        intrinsic = intrinsic[None, ...]
    assert intrinsic.shape[-2:] == (3,3), f"Bad intrinsic shape {intrinsic.shape}"
    return intrinsic

def ensure_depth_4d(depth_np):
    depth_np = np.array(depth_np)
    if depth_np.ndim == 2:
        return depth_np[None, ..., None]
    if depth_np.ndim == 3:
        return depth_np[..., None]
    if depth_np.ndim == 4:
        return depth_np
    raise ValueError(f"Unexpected depth shape {depth_np.shape}")

def apply_transform_pts(T_4x4, ptsNx3):
    R = T_4x4[:3,:3]
    t = T_4x4[:3,3]
    return (R @ ptsNx3.T).T + t

def batch_to_global_map(T_global_k, T_batch_k):
    """Rigid map batch-world → global using shared frame k."""
    return T_global_k @ np.linalg.inv(T_batch_k)

#  Load images
image_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*")))
assert image_paths, f" No images found in {IMAGE_DIR}"
num_frames = len(image_paths)
print(f"✓ Found {num_frames} images")

#  Load VGGT
print("\nLoading VGGT model...")
t0 = time.time()
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device).eval()
print(f"✓ Model loaded in {time.time()-t0:.1f}s")

#  Sliding-window planning
W = WINDOW_SIZE
O = WINDOW_OVERLAP
assert W > 0 and O >= 0 and O < W, "WINDOW_OVERLAP must be < WINDOW_SIZE"

windows = []
start = 0
while start < num_frames:
    end = min(start + W, num_frames)
    windows.append(list(range(start, end)))
    if end == num_frames:
        break
    start = end - O

print(f"✓ Planned {len(windows)} windows (size={W}, overlap={O})")

#  Global stores
all_pts, all_cols = [], []
global_extrinsics = {}
global_intrinsics = {}
first_global_set = False

# For saving in order at the end
extrinsics_global_list = [None] * num_frames
intrinsics_list        = [None] * num_frames

#  Inference per window
for wi, frame_indices in enumerate(windows, 1):
    print(f"\n▶ Window {wi}/{len(windows)}: frames {frame_indices[0]}–{frame_indices[-1]} (count={len(frame_indices)})")

    # Load & preprocess
    paths = [image_paths[i] for i in frame_indices]
    images = load_and_preprocess_images(paths, mode="pad")
    S, _, H, Wimg = images.shape

    images = images.to(device, dtype=torch.float32, non_blocking=True)

    with torch.no_grad():
        #  FP32 for aggregator + camera head
        with torch.autocast("cuda", enabled=False):
            batch = images.unsqueeze(0)
            tokens, ps_idx = model.aggregator(batch)
            pose_enc = model.camera_head(tokens)[-1]
            extrinsic_t, intrinsic_t = pose_encoding_to_extri_intri(pose_enc, (H, Wimg))

        #  Depth head in FP16 (optional)
        if USE_FP16_DEPTH:
            with torch.autocast("cuda", dtype=torch.float16):
                depth_map_t, depth_conf_t = model.depth_head(tokens, batch, ps_idx)
        else:
            with torch.autocast("cuda", enabled=False):
                depth_map_t, depth_conf_t = model.depth_head(tokens, batch, ps_idx)

    # Tensors → numpy
    extrinsic = ensure_4x4_extrinsics(to_numpy_squeezed(extrinsic_t))
    intrinsic = ensure_3x3_intrinsics(to_numpy_squeezed(intrinsic_t))

    depth_map  = ensure_depth_4d(np.squeeze(depth_map_t.detach().cpu().numpy()))
    depth_conf = np.squeeze(depth_conf_t.detach().cpu().numpy())
    if depth_conf.ndim == 2:
        depth_conf = depth_conf[None, ...]

    # Establish global frame
    if not first_global_set:
        k_local = 0
        global_extrinsics[frame_indices[k_local]] = extrinsic[k_local].copy()
        first_global_set = True
        print(f"  • Set global anchor at frame {frame_indices[k_local]}")
        T_global_from_batch = np.eye(4, dtype=np.float64)
    else:

        shared_id = None
        for li, fidx in enumerate(frame_indices):
            if fidx in global_extrinsics:
                shared_id = (li, fidx)
                break

        if shared_id is None:
            print("   No shared frame found — consider increasing WINDOW_OVERLAP.")
            T_global_from_batch = np.eye(4, dtype=np.float64)
        else:
            k_local, k_abs = shared_id
            T_global_from_batch = batch_to_global_map(
                global_extrinsics[k_abs],
                extrinsic[k_local]
            )
            print(f"   Aligning this window using shared frame {k_abs} (local {k_local})")


        extrinsic = (T_global_from_batch[None, ...] @ extrinsic)


    for si, abs_frame_idx in enumerate(frame_indices):

        global_extrinsics[abs_frame_idx] = extrinsic[si].copy()
        global_intrinsics[abs_frame_idx] = intrinsic[si].copy()
        extrinsics_global_list[abs_frame_idx] = extrinsic[si].copy()
        intrinsics_list[abs_frame_idx]       = intrinsic[si].copy()

        # Confidence mask
        mask = (depth_conf[si] >= CONF_THRES)


        pts_map = unproject_depth_map_to_point_map(
            depth_map[si][None, ...],
            extrinsic[si][None, ...],
            intrinsic[si][None, ...]
        )[0]

        pts = pts_map[mask]


        img_np = images[si].permute(1,2,0).detach().cpu().numpy()
        cols = (img_np * 255.0).astype(np.uint8)[mask]


        if pts.shape[0] > MAX_POINTS_FRAME:
            choice = np.random.choice(pts.shape[0], MAX_POINTS_FRAME, replace=False)
            pts, cols = pts[choice], cols[choice]

        all_pts.append(pts)
        all_cols.append(cols)
        print(f"  • Frame {abs_frame_idx:>4} → {pts.shape[0]:,} pts (mask≥{CONF_THRES})")

pts_all  = np.concatenate(all_pts, axis=0) if all_pts else np.zeros((0,3), dtype=np.float32)
cols_all = np.concatenate(all_cols, axis=0) if all_cols else np.zeros((0,3), dtype=np.uint8)
print(f"\n Total combined points before cleaning: {len(pts_all):,}")

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts_all.astype(np.float64))
pcd.colors = o3d.utility.Vector3dVector((cols_all.astype(np.float32) / 255.0))

# Outlier removal
if len(pcd.points) > 0:
    pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=RM_OUTLIER_K, std_ratio=RM_OUTLIER_STD)

# Voxel downsample
if len(pcd.points) > 0 and VOXEL_SIZE > 0:
    pcd = pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)

o3d.io.write_point_cloud(OUT_PLY, pcd, write_ascii=True)
print(f" Saved point cloud: {OUT_PLY}")

if SAVE_POSES_NPY:
    # Save in frame order for convenience
    E = np.stack(extrinsics_global_list, axis=0)
    K = np.stack(intrinsics_list, axis=0)
    np.save("camera_extrinsics_global.npy", E)
    np.save("camera_intrinsics.npy", K)
    print("✓ Saved camera_extrinsics_global.npy, camera_intrinsics.npy")

print("\n Reconstruction complete (global aligned)!")


✓ Using CUDA
✓ Found 5 images

Loading VGGT model...
✓ Model loaded in 34.1s
✓ Planned 1 windows (size=5, overlap=3)

▶ Window 1/1: frames 0–4 (count=5)
  • Set global anchor at frame 0
  • Frame    0 → 26,354 pts (mask≥7.0)
  • Frame    1 → 19,768 pts (mask≥7.0)
  • Frame    2 → 816 pts (mask≥7.0)
  • Frame    3 → 2,520 pts (mask≥7.0)
  • Frame    4 → 21 pts (mask≥7.0)

 Total combined points before cleaning: 49,479
 Saved point cloud: vggt_reconstruction_global_aligned.ply
✓ Saved camera_extrinsics_global.npy, camera_intrinsics.npy

 Reconstruction complete (global aligned)!


To delete images of directory...

In [10]:
# delete images after use

import os, shutil
img_dir = globals().get('IMAGE_DIR', 'images')
if os.path.isdir(img_dir):
    shutil.rmtree(img_dir); print(f" Deleted: {img_dir}")


 Deleted: images


Visualization...

In [11]:
import numpy as np
import open3d as o3d
import plotly.graph_objects as go

# Settings
PLY_PATH          = "vggt_reconstruction_global_aligned.ply"
MAX_POINTS_SHOW   = 120_000
HIDE_AXES         = True
ALIGN_GROUND      = True
PLANE_DIST_THRESH = 0.01
RANSAC_N          = 3
RANSAC_ITERS      = 1000

# Orbit camera (slow)
ENABLE_ROTATION   = True
ORBIT_FRAMES      = 240
ORBIT_RADIUS      = 1.8
ORBIT_Z           = 0.9
ORBIT_MS_PER_F    = 80

MARKER_SIZE       = 1
ALPHA             = 0.95

#  Load
pcd = o3d.io.read_point_cloud(PLY_PATH)
pts = np.asarray(pcd.points)
cols = (np.asarray(pcd.colors) * 255).astype(np.uint8)
print(f"✓ Loaded {len(pts):,} points from {PLY_PATH}")

# Ground align (optional)
R_align = np.eye(3)
t_align = np.zeros(3)

if ALIGN_GROUND and len(pts) > 1000:
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=PLANE_DIST_THRESH,
        ransac_n=RANSAC_N,
        num_iterations=RANSAC_ITERS
    )
    a, b, c, d = plane_model
    n = np.array([a, b, c], dtype=np.float64)
    n = n / (np.linalg.norm(n) + 1e-12)


    if n[2] < 0:
        n = -n

    ez = np.array([0.0, 0.0, 1.0], dtype=np.float64)
    v = np.cross(n, ez)
    s = np.linalg.norm(v)
    c_ = float(np.dot(n, ez))
    if s < 1e-8:
        R_align = np.eye(3)
    else:
        vx = np.array([[0, -v[2], v[1]],
                       [v[2], 0, -v[0]],
                       [-v[1], v[0], 0]], dtype=np.float64)
        R_align = np.eye(3) + vx + vx @ vx * ((1 - c_) / (s**2))

    pts = (R_align @ pts.T).T

    ground_z = np.median(pts[inliers, 2])
    pts[:, 2] -= ground_z
    print(f"✓ Ground aligned: normal→+Z, plane z shifted by {ground_z:.4f}")

# Subsample
if len(pts) > MAX_POINTS_SHOW:
    step = max(1, len(pts) // MAX_POINTS_SHOW)
    pts  = pts[::step]
    cols = cols[::step]
print(f"Visualizing {len(pts):,} points...")

# Figure
fig = go.Figure(go.Scatter3d(
    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
    mode='markers',
    marker=dict(
        size=MARKER_SIZE,
        color=['rgb(%d,%d,%d)' % tuple(c) for c in cols],
        opacity=ALPHA
    )
))

scene_dict = dict(
    aspectmode='data',
    xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
    xaxis=dict(showgrid=True, zeroline=False, backgroundcolor='rgba(0,0,0,0)'),
    yaxis=dict(showgrid=True, zeroline=False, backgroundcolor='rgba(0,0,0,0)'),
    zaxis=dict(showgrid=True, zeroline=False, backgroundcolor='rgba(0,0,0,0)'),
)

if HIDE_AXES:
    scene_dict.update(
        xaxis=dict(visible=False, showgrid=False, zeroline=False, showticklabels=False, showbackground=False),
        yaxis=dict(visible=False, showgrid=False, zeroline=False, showticklabels=False, showbackground=False),
        zaxis=dict(visible=False, showgrid=False, zeroline=False, showticklabels=False, showbackground=False),
    )

fig.update_layout(
    height=720,
    scene=scene_dict,
    margin=dict(l=0, r=0, b=0, t=32),
    title="VGGT 3D Reconstruction (Ground-aligned)"
)

fig.update_layout(scene_camera=dict(eye=dict(x=ORBIT_RADIUS, y=0.0, z=ORBIT_Z)))

#  Slow orbit you can pause any time
if ENABLE_ROTATION:
    T = np.linspace(0, 2*np.pi, ORBIT_FRAMES, endpoint=False)
    frames = [
        go.Frame(
            layout=dict(
                scene_camera=dict(
                    eye=dict(
                        x=ORBIT_RADIUS*np.cos(t),
                        y=ORBIT_RADIUS*np.sin(t),
                        z=ORBIT_Z
                    )
                )
            )
        ) for t in T
    ]
    fig.frames = frames

    fig.update_layout(
        updatemenus=[dict(
            type="buttons", showactive=False,
            x=0.015, y=0.02, xanchor="left", yanchor="bottom",
            buttons=[
                dict(
                    label="▶ Auto-rotate (slow)",
                    method="animate",
                    args=[None, {
                        "frame": {"duration": ORBIT_MS_PER_F, "redraw": False},
                        "fromcurrent": True,
                        "mode": "immediate",
                        "transition": {"duration": 0},
                    }]
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}]
                ),
            ]
        )]
    )

fig.show()


✓ Loaded 44,498 points from vggt_reconstruction_global_aligned.ply
✓ Ground aligned: normal→+Z, plane z shifted by 0.4298
Visualizing 44,498 points...
